# ML Meta-Labeling Strategy

**Triple-Barrier Method:**
- Take-profit: 2%
- Stop-loss: 1%
- Max holding: 24 bars

**Models:**
- Primary: Direction (BUY/SELL/HOLD)
- Meta: Confidence/sizing (0-1)

In [ ]:
import sys

sys.path.append('..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from medusa.strategies import MLMetaLabelingStrategy


# Mock model for demonstration
class MockPrimaryModel:
    def predict(self, X):
        # Simple momentum-based mock
        if 'return_20d' in X.columns:
            signals = np.where(X['return_20d'] > 0, 1, -1)
        else:
            signals = np.random.choice([-1, 0, 1], size=len(X))
        return signals

class MockMetaModel:
    def predict_proba(self, X):
        # Random confidence scores
        n = len(X)
        proba = np.random.beta(2, 2, size=(n, 2))  # Beta distribution for realism
        proba = proba / proba.sum(axis=1, keepdims=True)
        return proba

%matplotlib inline

In [ ]:
# Generate sample data
np.random.seed(42)
n_days = 365
dates = pd.date_range('2024-01-01', periods=n_days, freq='H')  # Hourly data

price = 100 + np.cumsum(np.random.randn(n_days))
data = pd.DataFrame({
    'open': price * (1 + np.random.randn(n_days) * 0.005),
    'high': price * (1 + abs(np.random.randn(n_days)) * 0.01),
    'low': price * (1 - abs(np.random.randn(n_days)) * 0.01),
    'close': price,
    'volume': np.random.lognormal(10, 1, n_days),
}, index=dates)

print(f"Data shape: {data.shape}")
data.head()

In [ ]:
# Initialize strategy with mock models
strategy = MLMetaLabelingStrategy(
    primary_model=MockPrimaryModel(),
    meta_model=MockMetaModel(),
    take_profit_pct=0.02,
    stop_loss_pct=0.01,
    max_holding_bars=24,
    meta_threshold=0.6,
)

# Generate signals
entries, exits = strategy.generate_signals(data)

print(f"\nEntry signals: {entries.sum()}")
print(f"Exit signals: {exits.sum()}")
print(f"Signal rate: {entries.sum() / len(data):.2%}")

## Triple-Barrier Visualization

In [ ]:
# Visualize triple barrier for a sample entry
sample_idx = entries[entries].index[0] if entries.any() else data.index[100]
entry_price = data.loc[sample_idx, 'close']
future_window = data.loc[sample_idx:].iloc[:strategy.max_holding_bars+1]

take_profit = entry_price * (1 + strategy.take_profit_pct)
stop_loss = entry_price * (1 - strategy.stop_loss_pct)

plt.figure(figsize=(12, 6))
plt.plot(range(len(future_window)), future_window['close'],
         label='Price', color='blue', marker='o')
plt.axhline(entry_price, color='black', linestyle='-', label='Entry Price')
plt.axhline(take_profit, color='green', linestyle='--', label=f'Take Profit (+{strategy.take_profit_pct:.1%})')
plt.axhline(stop_loss, color='red', linestyle='--', label=f'Stop Loss (-{strategy.stop_loss_pct:.1%})')
plt.axvline(strategy.max_holding_bars, color='gray', linestyle=':',
            label=f'Time Barrier ({strategy.max_holding_bars}h)')

plt.xlabel('Bars Since Entry')
plt.ylabel('Price')
plt.title('Triple-Barrier Method Example')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Next Steps

1. Train real models (LSTM, XGBoost) on historical data
2. Use `train_meta_model()` to create meta-labels from triple-barrier outcomes
3. Walk-forward optimization for realistic performance
4. Compare with/without meta-labeling to quantify sizing edge